In [1]:
# ==========================================
# LIVE DEMO - MODEL LOADING
# ==========================================

import os
import joblib
import numpy as np
import pandas as pd

print("=" * 70)
print("LIVE NETWORK INTRUSION DETECTION DEMO")
print("=" * 70)

# Search project for saved model files
model_files = []

for root, dirs, files in os.walk("."):

    for file in files:

        if file.lower().endswith(
            (".pkl", ".joblib")
        ):

            model_files.append(
                os.path.join(root, file)
            )

print("\nSaved model files found:")

for file in model_files:
    print("-", file)

LIVE NETWORK INTRUSION DETECTION DEMO

Saved model files found:
- .\exp_2_result\experiment2_hybrid_config.pkl
- .\exp_2_result\experiment2_isolation_forest.pkl
- .\exp_2_result\experiment2_isolation_scaler.pkl
- .\exp_2_result\experiment2_random_forest.pkl
- .\exp_2_result\experiment2_train_medians.pkl
- .\models\Supervised_rf_model.pkl
- .\models\unsupervised_iso_model.pkl
- .\scaled\fitted_scaler.pkl


In [2]:
# ==========================================
# LOAD RANDOM FOREST MODEL CORRECTLY
# ==========================================

import os
import joblib

print("=" * 70)
print("SEARCHING FOR RANDOM FOREST MODEL")
print("=" * 70)

rf_candidates = []

for file in model_files:

    filename = os.path.basename(file).lower()

    # Isolation Forest ko deliberately exclude karna hai
    if (
        ("rf" in filename or
         "random_forest" in filename or
         "randomforest" in filename)
        and "isolation" not in filename
    ):
        rf_candidates.append(file)

print("\nRandom Forest candidates:")

for file in rf_candidates:
    print("-", file)

if not rf_candidates:
    raise FileNotFoundError(
        "Random Forest model not found. "
        "Check the model filename/path."
    )

RF_MODEL_PATH = rf_candidates[0]

loaded_model = joblib.load(RF_MODEL_PATH)

print("\nLoaded model:")
print(type(loaded_model).__name__)
print("Path:", RF_MODEL_PATH)

# Safety check
if "RandomForest" not in type(loaded_model).__name__:

    raise TypeError(
        f"Wrong model loaded: {type(loaded_model).__name__}"
    )

rf_model = loaded_model

print("\nRandom Forest loaded successfully.")

SEARCHING FOR RANDOM FOREST MODEL

Random Forest candidates:
- .\exp_2_result\experiment2_random_forest.pkl
- .\models\Supervised_rf_model.pkl

Loaded model:
RandomForestClassifier
Path: .\exp_2_result\experiment2_random_forest.pkl

Random Forest loaded successfully.


In [3]:
# ==========================================
# VERIFY RANDOM FOREST
# ==========================================

print("Model type:", type(rf_model).__name__)

if hasattr(rf_model, "feature_names_in_"):

    print(
        "Expected features:",
        len(rf_model.feature_names_in_)
    )

    print(
        "First 10 features:",
        list(rf_model.feature_names_in_[:10])
    )

else:

    print(
        "WARNING: This model does not contain feature names."
    )

Model type: RandomForestClassifier
Expected features: 67
First 10 features: ['Protocol', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Fwd Packets Length Total', 'Bwd Packets Length Total', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std']


In [4]:
# ==========================================
# LOAD TEST DATA
# ==========================================

parquet_files = []

for root, dirs, files in os.walk("."):

    for file in files:

        if file.lower().endswith(".parquet"):

            parquet_files.append(
                os.path.join(root, file)
            )

print("=" * 70)
print("PARQUET TEST DATA")
print("=" * 70)

for file in parquet_files[:20]:
    print("-", file)

print(
    "\nTotal parquet files:",
    len(parquet_files)
)

PARQUET TEST DATA
- .\customer data\Benign-Monday-no-metadata.parquet
- .\customer data\Botnet-Friday-no-metadata.parquet
- .\customer data\Bruteforce-Tuesday-no-metadata.parquet
- .\customer data\DDoS-Friday-no-metadata.parquet
- .\customer data\DoS-Wednesday-no-metadata.parquet
- .\customer data\Infiltration-Thursday-no-metadata.parquet
- .\customer data\Portscan-Friday-no-metadata.parquet
- .\customer data\WebAttacks-Thursday-no-metadata.parquet
- .\data\Benign-Monday-no-metadata.parquet
- .\data\Botnet-Friday-no-metadata.parquet
- .\data\Bruteforce-Tuesday-no-metadata.parquet
- .\data\DDoS-Friday-no-metadata.parquet
- .\data\DoS-Wednesday-no-metadata.parquet
- .\data\Infiltration-Thursday-no-metadata.parquet
- .\data\Portscan-Friday-no-metadata.parquet
- .\data\WebAttacks-Thursday-no-metadata.parquet
- .\data_2018\Botnet-Friday-02-03-2018_TrafficForML_CICFlowMeter.parquet
- .\data_2018\Bruteforce-Wednesday-14-02-2018_TrafficForML_CICFlowMeter.parquet
- .\data_2018\DDoS1-Tuesday-20-

In [5]:
# ==========================================
# LIVE INFERENCE FUNCTION
# ==========================================

def live_test(sample, sample_number="Manual"):

    # Convert to DataFrame
    if isinstance(sample, pd.Series):

        sample_df = sample.to_frame().T

    else:

        sample_df = pd.DataFrame(sample)

    # Keep only features expected by model
    expected_features = rf_model.feature_names_in_

    missing_features = [
        col
        for col in expected_features
        if col not in sample_df.columns
    ]

    if missing_features:

        raise ValueError(
            f"Missing features: {missing_features}"
        )

    sample_df = sample_df[
        expected_features
    ]

    # Convert numeric
    sample_df = sample_df.apply(
        pd.to_numeric,
        errors="coerce"
    )

    # Handle invalid values
    sample_df = sample_df.replace(
        [np.inf, -np.inf],
        np.nan
    )

    sample_df = sample_df.fillna(0)

    # Prediction
    prediction = int(
        rf_model.predict(sample_df)[0]
    )

    probability = float(
        rf_model.predict_proba(
            sample_df
        )[0, 1]
    )

    # Severity
    if probability < 0.30:

        severity = "🟢 LOW RISK"

    elif probability < 0.70:

        severity = "🟡 MEDIUM RISK"

    else:

        severity = "🔴 HIGH RISK"

    decision = (
        "🚨 ATTACK DETECTED"
        if prediction == 1
        else "✅ BENIGN TRAFFIC"
    )

    print("=" * 70)
    print("LIVE INTRUSION DETECTION RESULT")
    print("=" * 70)

    print("Test Sample:", sample_number)

    print(
        f"Attack Probability : {probability:.4f}"
    )

    print(
        f"Risk Level         : {severity}"
    )

    print(
        f"Final Decision     : {decision}"
    )

    print("=" * 70)

    return {
        "prediction": prediction,
        "attack_probability": probability,
        "severity": severity,
        "decision": decision
    }

In [6]:
# ==========================================
# SELECT TEST SAMPLE
# ==========================================

# Use the first available parquet file
TEST_FILE = parquet_files[0]

test_df = pd.read_parquet(
    TEST_FILE
)

print("Test file:")
print(TEST_FILE)

print("\nShape:")
print(test_df.shape)

print("\nColumns:")
print(test_df.columns.tolist())

Test file:
.\customer data\Benign-Monday-no-metadata.parquet

Shape:
(458831, 78)

Columns:
['Protocol', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Fwd Packets Length Total', 'Bwd Packets Length Total', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Packet Length Min', 'Packet Length Max', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag

In [7]:
# ==========================================
# RUN LIVE TEST
# ==========================================

sample_index = 50

sample = test_df.iloc[sample_index]

result = live_test(
    sample,
    sample_number=sample_index
)

print("\nReturned Result:")
print(result)

LIVE INTRUSION DETECTION RESULT
Test Sample: 50
Attack Probability : 0.0000
Risk Level         : 🟢 LOW RISK
Final Decision     : ✅ BENIGN TRAFFIC

Returned Result:
{'prediction': 0, 'attack_probability': 0.0, 'severity': '🟢 LOW RISK', 'decision': '✅ BENIGN TRAFFIC'}
